# Experiment 21 — V3 (KCDP) Prompt Ablation

Quantifies each V3 prompt component's contribution by running 3 ablation
variants + a re-run baseline, scored against the existing 2-rater human gold
standard for the 10-student validation cohort.

**Only the prompt changes between conditions.** Model (gemini-2.5-flash),
temperature (0.3), parser, student/problem set, required-KC source, and the
scorer (`utils.metrics`) are all reused unchanged from exp19 / the V3 eval.

| Condition | rules_mode | kc_mode |
|---|---|---|
| baseline | full (14 rules) | per_problem |
| no_rules | none (0 rules)  | per_problem |
| reduced  | reduced (4 rules) | per_problem |
| no_kc    | full (14 rules) | full_vocab |

`reduced` keeps 4 disambiguation rules: the three `X vs If/Else` rules plus
`LogicCompareNum vs LogicAndNotOr` (indices 0,1,2,12 in `lib.v3_prompt`).

**Run order:** run the baseline cell first, confirm the sanity check
(~F1 0.839, κ 0.557, AC1 0.953), then run the remaining three conditions.

In [1]:
import json
import os
import sys
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
from google import genai
from google.genai import types

ROOT = Path("/mnt/d/Projects/kintsugi")
sys.path.insert(0, str(ROOT))

from lib.v3_prompt import build_v3_prompt
from utils.constants import GEMINI_API_KEY
from utils.metrics import evaluate_llm_vs_humans, KC_COLUMNS

API_KEY = GEMINI_API_KEY or os.environ.get("GOOGLE_API_KEY")

# --- Configuration (identical to exp19) ---
MODEL_ID = "gemini-2.5-flash"
SLEEP_SECONDS = 7          # Free tier: 7s. Paid tier: 3s.
TEMPERATURE = 0.3

STUDENT_IDS = [10155, 9948, 14189, 14352, 14362, 14363, 14374, 14414, 14474, 14499]

INPUT_DIR = ROOT / "scripts" / "annotation_tool" / "annotation_inputs"
HUMAN_DIR = ROOT / "dataset" / "Rater_KC_Tags" / "Rated_KC_V3"
OUTPUT_DIR = ROOT / "results" / "human_validation" / "ablation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# condition -> (rules_mode, kc_mode)
CONDITIONS = {
    "baseline": ("full", "per_problem"),
    "no_rules": ("none", "per_problem"),
    "reduced":  ("reduced", "per_problem"),
    "no_kc":    ("full", "full_vocab"),
}
# human-readable labels for the results table
CONDITION_LABELS = {
    "baseline": ("per_problem", "full (14)"),
    "no_rules": ("per_problem", "none (0)"),
    "reduced":  ("per_problem", "reduced (4)"),
    "no_kc":    ("full_vocab", "full (14)"),
}

VALID_KCS = set(KC_COLUMNS)

pp_df = pd.read_csv(ROOT / "dataset" / "CodeWorkout" / "Problem_Prompts" / "problem_prompts.csv")

def get_required_kcs(problem_id):
    row = pp_df[pp_df["ProblemID"] == problem_id]
    if row.empty:
        return []
    row = row.iloc[0]
    return [kc for kc in KC_COLUMNS if pd.notna(row.get(kc)) and float(row.get(kc)) == 1.0]

def get_problem_info(problem_id):
    row = pp_df[pp_df["ProblemID"] == problem_id]
    if row.empty:
        return None, None
    row = row.iloc[0]
    return row["Requirement"], int(row["AssignmentID"])

print("Project root:", ROOT)
print("Model:", MODEL_ID, "| temp:", TEMPERATURE)
print("Conditions:", list(CONDITIONS))
print("Output dir:", OUTPUT_DIR)

Project root: /mnt/d/Projects/kintsugi
Model: gemini-2.5-flash | temp: 0.3
Conditions: ['baseline', 'no_rules', 'reduced', 'no_kc']
Output dir: /mnt/d/Projects/kintsugi/results/human_validation/ablation


In [2]:
# --- Gemini client (identical to exp19) ---
if not API_KEY:
    raise ValueError("Set GEMINI_API_KEY or GOOGLE_API_KEY before running Experiment 21.")

client = genai.Client(api_key=API_KEY)

def call_gemini(prompt_text):
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt_text,
        config=types.GenerateContentConfig(temperature=TEMPERATURE),
    )
    return response.text or ""

print("Gemini client configured.")

Gemini client configured.


In [3]:
# --- Load student data + runtime estimate ---
student_data = {}
for sid in STUDENT_IDS:
    with open(INPUT_DIR / f"student_{sid}.json") as f:
        student_data[sid] = json.load(f)

calls_per_condition = sum(
    sum(1 for v in d["submissions"].values() if v["score"] < 1.0)
    for d in student_data.values()
)
print(f"Calls per condition: {calls_per_condition}")
print(f"All 4 conditions: {calls_per_condition * 4} calls")
print(f"~{calls_per_condition * 4 * SLEEP_SECONDS / 60:.0f} min total at {SLEEP_SECONDS}s sleep")

Calls per condition: 188
All 4 conditions: 752 calls
~88 min total at 7s sleep


In [4]:
# --- Parser (verbatim from exp19) ---
def parse_llm_response(raw_text):
    """Parse Gemini JSON response. Returns (parsed_dict, status)."""
    cleaned = (raw_text or "").strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    if cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()
    try:
        return json.loads(cleaned), "ok"
    except json.JSONDecodeError:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start != -1 and end != -1 and start < end:
            try:
                return json.loads(cleaned[start:end + 1]), "ok_extracted_json"
            except json.JSONDecodeError as e:
                return {"reasoning": raw_text, "knowledge_gaps": []}, f"parse_error: {e}"
        return {"reasoning": raw_text, "knowledge_gaps": []}, "parse_error: no_json_object"

def clean_gaps(parsed):
    gaps = parsed.get("knowledge_gaps", [])
    if isinstance(gaps, str):
        gaps = [gaps]
    if not isinstance(gaps, list):
        return []
    return [g for g in gaps if g in VALID_KCS]

In [5]:
# --- Run loop (one condition at a time; per-student checkpointing) ---
def run_condition(condition):
    rules_mode, kc_mode = CONDITIONS[condition]
    print(f"\n{'#'*70}\n# CONDITION {condition}  (rules_mode={rules_mode}, kc_mode={kc_mode})\n{'#'*70}")

    for idx, sid in enumerate(STUDENT_IDS):
        out_path = OUTPUT_DIR / f"llm_ablation_{condition}_{sid}.json"
        if out_path.exists():
            print(f"[{idx+1}/10] Student {sid} — ALREADY DONE, skipping.")
            continue

        print(f"\n{'='*60}\n[{idx+1}/10] {condition} | Student {sid}\n{'='*60}")
        submissions = student_data[sid]["submissions"]
        annotations, raw_responses, errors = {}, {}, []
        call_count = 0

        for pid_str in sorted(submissions.keys(), key=lambda x: int(x)):
            pid = int(pid_str)
            sub = submissions[pid_str]
            score = sub["score"]

            if score >= 1.0:
                annotations[pid_str] = {"gaps": []}
                continue

            requirement, assignment_id = get_problem_info(pid)
            required_kcs = get_required_kcs(pid)
            if requirement is None:
                print(f"  WARNING: Problem {pid} not in problem_prompts.csv, skipping.")
                annotations[pid_str] = {"gaps": []}
                raw_responses[pid_str] = {"error": "problem_not_found"}
                continue

            prompt = build_v3_prompt(
                problem_id=pid, requirement=requirement, assignment_id=assignment_id,
                required_kcs=required_kcs, student_code=sub["code"], score=score,
                rules_mode=rules_mode, kc_mode=kc_mode,
            )

            call_count += 1
            try:
                t0 = time.time()
                raw_response = call_gemini(prompt)
                elapsed = time.time() - t0
                parsed, status = parse_llm_response(raw_response)
                gaps = clean_gaps(parsed)
                annotations[pid_str] = {"gaps": gaps}
                raw_responses[pid_str] = {
                    "raw_response": raw_response,
                    "parsed_response": parsed,
                    "parse_status": status,
                    "time_sec": round(elapsed, 3),
                    "score": score,
                    "assignment_id": assignment_id,
                    "required_kcs": required_kcs,
                    "invalid_kcs": [g for g in parsed.get("knowledge_gaps", []) if g not in VALID_KCS]
                    if isinstance(parsed.get("knowledge_gaps", []), list) else [],
                }
                print(f"  P{pid} (score={score:.2f}) -> [{', '.join(gaps) if gaps else '(none)'}] ({elapsed:.1f}s) [{status}]")
            except Exception as e:
                print(f"  ERROR on P{pid}: {e}")
                errors.append(f"API error on P{pid}: {e}")
                annotations[pid_str] = {"gaps": []}
                raw_responses[pid_str] = {"error": str(e), "score": score,
                                          "assignment_id": assignment_id, "required_kcs": required_kcs}
            time.sleep(SLEEP_SECONDS)

        result = {
            "rater": f"LLM_Gemini_V3_ablation_{condition}",
            "condition": condition, "rules_mode": rules_mode, "kc_mode": kc_mode,
            "studentId": str(sid), "student_id": str(sid),
            "model_id": MODEL_ID, "temperature": TEMPERATURE,
            "exportDate": datetime.now().isoformat(),
            "total_problems": len(submissions), "totalAnnotated": len(annotations),
            "total_calls": call_count, "errors": errors,
            "annotations": annotations, "raw_responses": raw_responses,
        }
        with open(out_path, "w") as f:
            json.dump(result, f, indent=2)
        n_gaps = sum(1 for a in annotations.values() if a.get("gaps"))
        print(f"  SAVED {out_path.name} | calls={call_count} with_gaps={n_gaps} errors={len(errors)}")

    print(f"\nCondition {condition} complete.")

## Step 1 — Run the baseline (re-run, do not trust old numbers)

In [6]:
run_condition("baseline")


######################################################################
# CONDITION baseline  (rules_mode=full, kc_mode=per_problem)
######################################################################
[1/10] Student 10155 — ALREADY DONE, skipping.
[2/10] Student 9948 — ALREADY DONE, skipping.
[3/10] Student 14189 — ALREADY DONE, skipping.
[4/10] Student 14352 — ALREADY DONE, skipping.
[5/10] Student 14362 — ALREADY DONE, skipping.
[6/10] Student 14363 — ALREADY DONE, skipping.
[7/10] Student 14374 — ALREADY DONE, skipping.
[8/10] Student 14414 — ALREADY DONE, skipping.
[9/10] Student 14474 — ALREADY DONE, skipping.
[10/10] Student 14499 — ALREADY DONE, skipping.

Condition baseline complete.


## Scoring helpers (reuse `utils.metrics` — the exact V3 scorer)

In [7]:
def find_one(pattern, directory):
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matched {pattern} in {directory}")
    return matches[-1]

def normalize_gaps(value):
    if isinstance(value, dict):
        gaps = value.get("gaps", [])
    elif isinstance(value, list):
        gaps = value
    else:
        gaps = []
    if not isinstance(gaps, list):
        return set()
    return {g for g in gaps if g in VALID_KCS}

def load_annotation_file(path):
    with path.open(encoding="utf-8") as f:
        data = json.load(f)
    sid = str(data.get("studentId", data.get("student_id", "unknown")))
    return sid, {f"{sid}_{pid}": normalize_gaps(v) for pid, v in data.get("annotations", {}).items()}

def merge_files(file_map):
    merged = {}
    for expected_sid, path in file_map.items():
        loaded_sid, anns = load_annotation_file(path)
        if loaded_sid != expected_sid:
            raise ValueError(f"Expected {expected_sid}, found {loaded_sid} in {path.name}")
        merged.update(anns)
    return merged

def load_condition(condition):
    return merge_files({str(sid): OUTPUT_DIR / f"llm_ablation_{condition}_{sid}.json" for sid in STUDENT_IDS})

def parse_rate(condition):
    """Fraction of LLM-called problems (those with a parse_status) that parsed validly."""
    total = ok = 0
    for sid in STUDENT_IDS:
        data = json.loads((OUTPUT_DIR / f"llm_ablation_{condition}_{sid}.json").read_text())
        for rec in data.get("raw_responses", {}).values():
            status = rec.get("parse_status")
            if status is None:
                continue
            total += 1
            ok += status.startswith("ok")
    return (ok / total) if total else float("nan")

# Human gold standard (2 raters)
human_a = merge_files({str(sid): find_one(f"kc_annotations_Pranay Ghuge_{sid}_*.json", HUMAN_DIR) for sid in STUDENT_IDS})
human_b = merge_files({str(sid): find_one(f"kc_annotations_Arundhati Das_{sid}_*.json", HUMAN_DIR) for sid in STUDENT_IDS})

def common_items_for(reference_condition="baseline"):
    """Common problem set: human_a ∩ human_b ∩ reference LLM run (same definition as V3 eval)."""
    ref = load_condition(reference_condition)
    return sorted(set(human_a) & set(human_b) & set(ref),
                  key=lambda x: (int(x.split("_")[0]), int(x.split("_")[1])))

def score_condition(condition, common):
    llm = load_condition(condition)
    missing = [k for k in common if k not in llm]
    if missing:
        raise ValueError(f"{condition} missing {len(missing)} items, e.g. {missing[:3]}")
    avg = evaluate_llm_vs_humans(human_a, human_b, llm, common, KC_COLUMNS)[3]  # AvgHuman vs LLM
    return avg["Problem_F1"], avg["Cohen_kappa"], avg["Gwet_AC1"], parse_rate(condition)

## Sanity check — baseline must reproduce ~F1 0.839, κ 0.557, AC1 0.953

If it doesn't, STOP and investigate before running the other conditions.
(Temperature is 0.3, so small drift from the stored V3 numbers is expected.)

In [8]:
common = common_items_for("baseline")
print(f"Common problem annotations: {len(common)}")

f1, k, ac1, prate = score_condition("baseline", common)
print(f"\nBaseline (re-run): F1={f1:.3f}  κ={k:.3f}  AC1={ac1:.3f}  parse={prate*100:.1f}%")
print("Expected (stored V3): F1≈0.839  κ≈0.557  AC1≈0.953")

ok = abs(f1 - 0.839) < 0.03 and abs(k - 0.557) < 0.05 and abs(ac1 - 0.953) < 0.02
print("\nSANITY", "PASS — proceed to the other conditions." if ok else "FAIL — investigate before continuing.")
assert ok, "Baseline did not reproduce the stored V3 numbers within tolerance — STOP and investigate before running other conditions."

Common problem annotations: 372

Baseline (re-run): F1=0.831  κ=0.548  AC1=0.953  parse=96.3%
Expected (stored V3): F1≈0.839  κ≈0.557  AC1≈0.953

SANITY PASS — proceed to the other conditions.


## Step 2 — Run the remaining three conditions

Only runs after the baseline sanity check passes.

In [9]:
for cond in ["no_rules", "reduced", "no_kc"]:
    run_condition(cond)


######################################################################
# CONDITION no_rules  (rules_mode=none, kc_mode=per_problem)
######################################################################
[1/10] Student 10155 — ALREADY DONE, skipping.
[2/10] Student 9948 — ALREADY DONE, skipping.

[3/10] no_rules | Student 14189
  P3 (score=0.88) -> [If/Else] (20.1s) [ok]
  P13 (score=0.96) -> [LogicCompareNum] (36.7s) [ok_extracted_json]
  P22 (score=0.36) -> [DefFunction, LogicCompareNum, If/Else] (22.9s) [ok]
  P24 (score=0.86) -> [LogicCompareNum, LogicAndNotOr] (35.6s) [ok]
  P25 (score=0.76) -> [LogicAndNotOr] (24.3s) [ok]
  P28 (score=0.60) -> [If/Else, LogicCompareNum, StringFormat, StringConcat, StringIndex, StringLen] (14.8s) [ok]
  P32 (score=0.00) -> [(none)] (2.2s) [ok]
  P34 (score=0.00) -> [(none)] (3.6s) [ok]
  P36 (score=0.41) -> [(none)] (2.3s) [ok]
  P37 (score=0.60) -> [(none)] (2.8s) [ok]
  P38 (score=0.47) -> [(none)] (2.0s) [ok]
  P39 (score=0.63) -> [(none)] (3.9

## Step 3 — Score every condition and emit the results table

In [10]:
import pandas as pd

common = common_items_for("baseline")  # same items for all conditions
rows = []
for cond in ["baseline", "no_rules", "reduced", "no_kc"]:
    f1, k, ac1, prate = score_condition(cond, common)
    kc_inj, rules = CONDITION_LABELS[cond]
    rows.append({"Condition": cond, "KC injection": kc_inj, "Rules": rules,
                 "F1": f1, "κ": k, "AC1": ac1, "Parse %": prate * 100})

df = pd.DataFrame(rows)

lines = ["| Condition | KC injection | Rules | F1 | κ | AC1 | Parse % |",
         "|---|---|---|---|---|---|---|"]
for r in rows:
    lines.append(f"| {r['Condition']} | {r['KC injection']} | {r['Rules']} | "
                 f"{r['F1']:.3f} | {r['κ']:.3f} | {r['AC1']:.3f} | {r['Parse %']:.1f}% |")
table = "\n".join(lines)
print(f"Scored over {len(common)} common problem annotations.\n")
print(table)

md_path = ROOT / "ablation_results.md"
md_path.write_text(
    "# V3 (KCDP) Prompt Ablation Results\n\n"
    f"Scored against the 2-rater human gold standard over {len(common)} common "
    "problem annotations (10 struggling students). Metrics are the AvgHuman-vs-LLM "
    "row from `utils.metrics.evaluate_llm_vs_humans` (the same scorer behind the V3 "
    "headline numbers). Model: gemini-2.5-flash, temperature 0.3. Only the prompt "
    "varies between conditions.\n\n"
    "Parse % = fraction of LLM-called problems with a valid `parsed_response` "
    "(perfect-score problems are skipped, not called).\n\n" + table + "\n"
)
print(f"\nWrote {md_path}")
df

Scored over 372 common problem annotations.

| Condition | KC injection | Rules | F1 | κ | AC1 | Parse % |
|---|---|---|---|---|---|---|
| baseline | per_problem | full (14) | 0.831 | 0.548 | 0.953 | 96.3% |
| no_rules | per_problem | none (0) | 0.847 | 0.571 | 0.952 | 98.9% |
| reduced | per_problem | reduced (4) | 0.829 | 0.533 | 0.949 | 96.3% |
| no_kc | full_vocab | full (14) | 0.820 | 0.496 | 0.950 | 98.4% |

Wrote /mnt/d/Projects/kintsugi/ablation_results.md


,Condition,KC injection,Rules,F1,κ,AC1,Parse %
0,baseline,per_problem,full (14),0.830803,0.547862,0.952740,96.276596
1,no_rules,per_problem,none (0),0.846883,0.570795,0.951735,98.936170
2,reduced,per_problem,reduced (4),0.829348,0.533353,0.949430,96.276596
3,no_kc,full_vocab,full (14),0.819919,0.495773,0.950380,98.404255
